# 🧠 Alzheimer's Disease Classification
### EfficientNetB3 + Transfer Learning | TensorFlow/Keras

**Dataset:** [Alzheimer's Multiclass Dataset (Equal & Augmented)](https://www.kaggle.com/datasets/aryansinghal10/alzheimers-multiclass-dataset-equal-and-augmented)

**Classes:**
- `NonDemented`
- `VeryMildDemented`
- `MildDemented`
- `ModerateDemented`

**Strategy:** Two-phase training
1. **Phase 1** — Freeze EfficientNetB3 backbone, train only the new classification head
2. **Phase 2** — Unfreeze top layers and fine-tune end-to-end at a lower learning rate


## ⚙️ Step 1: Setup — Install Kaggle API & Download Dataset

In [ ]:
# Install kaggle API
!pip install kaggle -q

# Upload your kaggle.json API token
# Go to: https://www.kaggle.com/settings → API → Create New Token
from google.colab import files
print("Upload your kaggle.json file:")
uploaded = files.upload()

In [ ]:
import os

# Set up Kaggle credentials
os.makedirs('/root/.config/kaggle', exist_ok=True)
!cp kaggle.json /root/.config/kaggle/
!chmod 600 /root/.config/kaggle/kaggle.json

# Download dataset
!kaggle datasets download -d aryansinghal10/alzheimers-multiclass-dataset-equal-and-augmented

# Unzip
!unzip -q alzheimers-multiclass-dataset-equal-and-augmented.zip -d alzheimers_data
!ls alzheimers_data

## 📦 Step 2: Imports & Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import random
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
)
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# ─── CONFIG ───────────────────────────────────────────────────────────────────
# Update DATA_DIR if your folder structure differs after unzipping
DATA_DIR      = Path("alzheimers_data")   # root of unzipped dataset
IMG_SIZE      = 224          # EfficientNetB3 default input size
BATCH_SIZE    = 32
EPOCHS_HEAD   = 15           # Phase 1: train head only
EPOCHS_FINETUNE = 25         # Phase 2: fine-tune top layers
LR_HEAD       = 1e-3
LR_FINETUNE   = 1e-5
SEED          = 42
CLASS_NAMES   = ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']

tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

## 🗂️ Step 3: Explore Dataset & Class Distribution

In [ ]:
# Dataset has a single flat folder: combined_images/<class>/
SOURCE_DIR = DATA_DIR / "combined_images"

print("📊 Class distribution (full dataset):")
total = 0
class_counts = {}
for cls in sorted(SOURCE_DIR.iterdir()):
    if cls.is_dir():
        n = len(list(cls.glob("*.jpg"))) + len(list(cls.glob("*.png")))
        class_counts[cls.name] = n
        print(f"  {cls.name:<25}: {n:>6} images")
        total += n
print(f"  {'TOTAL':<25}: {total:>6} images")

# Note the imbalance
print("\n⚠️  Note: classes are slightly imbalanced.")
print("   class_weight will be used during training to compensate.")

In [ ]:
import shutil

# ─── Split into train / val / test (70 / 15 / 15, stratified per class) ──────
TRAIN_DIR = DATA_DIR / "train"
VAL_DIR   = DATA_DIR / "val"
TEST_DIR  = DATA_DIR / "test"
VAL_SPLIT  = 0.15
TEST_SPLIT = 0.15

if TRAIN_DIR.exists():
    print("Split already done — skipping.")
else:
    print("Creating 70 / 15 / 15 train / val / test split...\n")
    for cls_dir in sorted(SOURCE_DIR.iterdir()):
        if not cls_dir.is_dir(): continue
        images = list(cls_dir.glob("*.jpg")) + list(cls_dir.glob("*.png"))
        random.shuffle(images)
        n      = len(images)
        n_val  = int(n * VAL_SPLIT)
        n_test = int(n * TEST_SPLIT)
        splits = {
            "val"  : images[:n_val],
            "test" : images[n_val : n_val + n_test],
            "train": images[n_val + n_test:],
        }
        for split_name, split_imgs in splits.items():
            dest = DATA_DIR / split_name / cls_dir.name
            dest.mkdir(parents=True, exist_ok=True)
            for img in split_imgs:
                shutil.copy(img, dest / img.name)
        print(f"  {cls_dir.name:<25}: "
              f"{len(splits['train'])} train | "
              f"{len(splits['val'])} val | "
              f"{len(splits['test'])} test")
    print("\n✅ Split complete.")

## 🖼️ Step 4: Visualise Sample Images

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle("Sample MRI Scans per Class", fontsize=16, fontweight='bold')

for col, cls in enumerate(sorted(SOURCE_DIR.iterdir())):
    if not cls.is_dir(): continue
    imgs = list(cls.glob("*.jpg")) + list(cls.glob("*.png"))
    for row in range(2):
        img_path = random.choice(imgs)
        img = plt.imread(str(img_path))
        axes[row][col].imshow(img, cmap='gray')
        axes[row][col].set_title(cls.name, fontsize=9)
        axes[row][col].axis('off')

plt.tight_layout()
plt.show()

## 🔄 Step 5: Data Pipelines with Augmentation

In [ ]:
# ─── Augmentation layer (applied only during training) ───────────────────────
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10),
    layers.RandomBrightness(0.10),
    layers.RandomContrast(0.10),
], name="augmentation")

# ─── Load datasets from the auto-split train/ and val/ folders ───────────────
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode='categorical'
)

CLASS_NAMES = train_ds.class_names
NUM_CLASSES = len(CLASS_NAMES)
print(f"Classes ({NUM_CLASSES}): {CLASS_NAMES}")

# ─── Class weights to handle imbalance ───────────────────────────────────────
# Counts are from the full dataset; weights are inversely proportional to frequency.
train_counts = {cls: class_counts[cls] for cls in CLASS_NAMES}
total_train  = sum(train_counts.values())
class_weight = {
    i: total_train / (NUM_CLASSES * count)
    for i, (cls, count) in enumerate(train_counts.items())
}
print("\nClass weights:")
for i, cls in enumerate(CLASS_NAMES):
    print(f"  [{i}] {cls:<25}: {class_weight[i]:.3f}")

# ─── Performance tuning ───────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

def prepare_train(ds):
    return (
        ds
        .map(lambda x, y: (data_augmentation(x, training=True), y),
             num_parallel_calls=AUTOTUNE)
        .prefetch(AUTOTUNE)
    )

def prepare_val(ds):
    return ds.prefetch(AUTOTUNE)

train_ds = prepare_train(train_ds)
val_ds   = prepare_val(val_ds)

## 🏗️ Step 6: Build the Model

In [ ]:
def build_model(num_classes: int, trainable_backbone: bool = False) -> Model:
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))

    # EfficientNetB3 includes its own preprocessing internally
    backbone = EfficientNetB3(
        include_top=False,
        weights="imagenet",
        input_tensor=inputs
    )
    backbone.trainable = trainable_backbone

    x = backbone.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    return model, backbone

# Phase 1 model: frozen backbone
model, backbone = build_model(NUM_CLASSES, trainable_backbone=False)

model.compile(
    optimizer=keras.optimizers.Adam(LR_HEAD),
    loss='categorical_crossentropy',
    metrics=['accuracy',
             keras.metrics.AUC(name='auc'),
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')]
)

model.summary(show_trainable=True)

## 🚀 Step 7: Phase 1 — Train Classification Head

In [ ]:
os.makedirs("checkpoints", exist_ok=True)

callbacks_phase1 = [
    ModelCheckpoint(
        "checkpoints/best_head.keras",
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, verbose=1
    ),
    CSVLogger("checkpoints/phase1_log.csv")
]

print("=" * 60)
print("PHASE 1: Training classification head (backbone frozen)")
print("=" * 60)

history1 = model.fit(
    train_ds,
    epochs=EPOCHS_HEAD,
    validation_data=val_ds,
    callbacks=callbacks_phase1
)

## 🔓 Step 8: Phase 2 — Fine-tune Top Layers

In [ ]:
# Unfreeze the top ~30% of backbone layers
backbone.trainable = True
fine_tune_from = int(len(backbone.layers) * 0.70)

for layer in backbone.layers[:fine_tune_from]:
    layer.trainable = False

print(f"Total backbone layers  : {len(backbone.layers)}")
print(f"Fine-tuning from layer : {fine_tune_from}")
print(f"Trainable layers       : {len(backbone.layers) - fine_tune_from}")

model.compile(
    optimizer=keras.optimizers.Adam(LR_FINETUNE),
    loss='categorical_crossentropy',
    metrics=['accuracy',
             keras.metrics.AUC(name='auc'),
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall')]
)

callbacks_phase2 = [
    ModelCheckpoint(
        "checkpoints/best_finetune.keras",
        monitor='val_accuracy', save_best_only=True, verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy', patience=7,
        restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.3, patience=3,
        min_lr=1e-7, verbose=1
    ),
    CSVLogger("checkpoints/phase2_log.csv")
]

print("\n" + "=" * 60)
print("PHASE 2: Fine-tuning top backbone layers")
print("=" * 60)

history2 = model.fit(
    train_ds,
    epochs=EPOCHS_FINETUNE,
    validation_data=val_ds,
    callbacks=callbacks_phase2
)

## 📈 Step 9: Plot Training Curves

In [ ]:
def merge_history(h1, h2):
    """Concatenate two Keras History objects."""
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history[key]
    return merged

history = merge_history(history1, history2)
phase1_end = len(history1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Training History", fontsize=16, fontweight='bold')

for ax, metric, title in zip(
    axes,
    ['accuracy', 'loss'],
    ['Accuracy', 'Loss']
):
    ax.plot(history[metric], label=f'Train {title}', color='steelblue')
    ax.plot(history[f'val_{metric}'], label=f'Val {title}', color='coral')
    ax.axvline(x=phase1_end - 0.5, color='gray', linestyle='--',
               label='Fine-tune starts')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 🧪 Step 10: Evaluate on Validation Set

In [ ]:
# Load best checkpoint
model = keras.models.load_model("checkpoints/best_finetune.keras")

# ─── Load the held-out TEST set ───────────────────────────────────────────────
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode='categorical'
).prefetch(tf.data.AUTOTUNE)

# ─── Evaluate on val set ──────────────────────────────────────────────────────
print("📊 Validation set metrics:")
val_results = model.evaluate(val_ds, verbose=1)

print("\n📊 Test set metrics (held-out):")
test_results = model.evaluate(test_ds, verbose=1)

# ─── Per-class report on test set ────────────────────────────────────────────
y_true, y_pred = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("\n📋 Classification Report (Test Set):")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# ─── Confusion Matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, data, fmt, title in zip(
    axes,
    [cm, cm_pct],
    ['d', '.1f'],
    ['Confusion Matrix — Test Set (counts)', 'Confusion Matrix — Test Set (%)']
):
    sns.heatmap(
        data, annot=True, fmt=fmt,
        xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
        cmap='Blues', linewidths=0.5, ax=ax
    )
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

## 🔍 Step 11: Visualise Predictions

In [ ]:
# Show a batch of predictions from the TEST set with confidence scores
sample_images, sample_labels = next(iter(test_ds))
sample_preds = model.predict(sample_images, verbose=0)

fig, axes = plt.subplots(4, 8, figsize=(20, 10))
fig.suptitle("Predictions (Green = Correct, Red = Wrong)", fontsize=14, fontweight='bold')

for i, ax in enumerate(axes.flat):
    if i >= len(sample_images):
        ax.axis('off')
        continue
    img = sample_images[i].numpy().astype('uint8')
    true_idx = np.argmax(sample_labels[i])
    pred_idx = np.argmax(sample_preds[i])
    confidence = sample_preds[i][pred_idx] * 100
    correct = true_idx == pred_idx

    ax.imshow(img, cmap='gray')
    ax.set_title(
        f"{CLASS_NAMES[pred_idx][:8]}\n{confidence:.0f}%",
        color='green' if correct else 'red',
        fontsize=7
    )
    ax.axis('off')

plt.tight_layout()
plt.show()

## 💾 Step 12: Save & Export Model

In [ ]:
# Save in Keras native format
model.save("alzheimers_efficientnetb3_final.keras")
print("✅ Model saved as alzheimers_efficientnetb3_final.keras")

# Save class names for inference
import json
with open("class_names.json", "w") as f:
    json.dump(CLASS_NAMES, f)
print("✅ Class names saved as class_names.json")

# Download to your computer (Google Colab)
from google.colab import files
files.download("alzheimers_efficientnetb3_final.keras")
files.download("class_names.json")

## 🔮 Step 13: Inference on a Single Image

In [ ]:
import json
from tensorflow.keras.preprocessing import image as keras_image

def predict_single_image(img_path: str, model, class_names: list):
    """
    Load a single MRI image and return the predicted class + confidence.
    img_path: path to a .jpg or .png file
    """
    img = keras_image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    img_array = keras_image.img_to_array(img)          # shape (224, 224, 3)
    img_array = np.expand_dims(img_array, axis=0)      # shape (1, 224, 224, 3)

    probs = model.predict(img_array, verbose=0)[0]     # shape (4,)
    pred_idx = np.argmax(probs)

    print(f"\nImage : {img_path}")
    print(f"Prediction : {class_names[pred_idx]} ({probs[pred_idx]*100:.1f}%)")
    print("\nAll probabilities:")
    for name, prob in zip(class_names, probs):
        bar = '█' * int(prob * 40)
        print(f"  {name:<22}: {bar} {prob*100:.1f}%")

    # Show image
    plt.figure(figsize=(4, 4))
    plt.imshow(keras_image.load_img(img_path), cmap='gray')
    plt.title(f"{class_names[pred_idx]} ({probs[pred_idx]*100:.1f}%)", fontsize=12)
    plt.axis('off')
    plt.show()

# ── Example usage (change path to any image) ──────────────────────────────────
# predict_single_image("path/to/your/mri_scan.jpg", model, CLASS_NAMES)
print("Ready. Call predict_single_image('your_image.jpg', model, CLASS_NAMES) to run inference.")